# LLM Quant Lab - Setup & Quickstart

This notebook helps you get started with the LLM Quant Lab quantization experiments.

## Prerequisites

1. **ROCm GPU**: AMD GPU with ROCm 6.x (MI250/MI300 series)
2. **Docker**: Docker with AMD GPU support
3. **HuggingFace Token**: For accessing gated models
4. **Weights & Biases**: For experiment tracking (optional)

## Setup Steps

1. Configure environment variables
2. Boot Docker containers
3. Verify GPU access
4. Download test model
5. Run quick validation

## 1. Environment Configuration

Edit the `.env` file in the project root with your credentials.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(".")
sys.path.insert(0, str(PROJECT_ROOT))

# Load environment
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

# Check required environment variables
required_vars = [
    "HIP_VISIBLE_DEVICES",
    "POSTGRES_USER",
    "POSTGRES_PASSWORD",
    "POSTGRES_DB",
]

optional_vars = [
    "HF_TOKEN",
    "WANDB_API_KEY",
    "SCIENTIST_LLM_API_KEY",
]

print("Environment Configuration Check")
print("=" * 50)

all_ok = True
for var in required_vars:
    value = os.getenv(var)
    status = "✓" if value else "✗ MISSING"
    if not value:
        all_ok = False
    print(f"  {var}: {status}")

print("\nOptional:")
for var in optional_vars:
    value = os.getenv(var)
    status = "✓ Set" if value else "○ Not set"
    print(f"  {var}: {status}")

if not all_ok:
    print("\n⚠️  Please configure missing variables in .env file")

: 

## 2. Verify GPU Access

In [ ]:
import torch

print("PyTorch Configuration")
print("=" * 50)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}: {props.name}")
        print(f"  Memory: {props.total_memory / 1e9:.1f} GB")
        print(f"  Compute capability: {props.major}.{props.minor}")
    
    # Quick GPU test
    print("\nGPU Test:")
    x = torch.randn(1000, 1000, device="cuda")
    y = torch.randn(1000, 1000, device="cuda")
    z = x @ y
    print(f"  Matrix multiply test: ✓ (output shape: {z.shape})")
else:
    print("\n⚠️  No GPU detected. Check ROCm installation.")

## 3. Test HuggingFace Access

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Test with a small public model
test_model = "facebook/opt-125m"

print(f"Testing HuggingFace access with {test_model}")
print("=" * 50)

try:
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(test_model)
    print("  ✓ Tokenizer loaded")
    
    print("Loading model (this may take a moment)...")
    model = AutoModelForCausalLM.from_pretrained(
        test_model,
        torch_dtype=torch.float16,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    print("  ✓ Model loaded")
    
    # Quick inference test
    print("\nRunning inference test...")
    inputs = tokenizer("The capital of France is", return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=10)
    
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"  Output: {result}")
    print("  ✓ Inference test passed")
    
    # Model info
    num_params = sum(p.numel() for p in model.parameters())
    print(f"\nModel info:")
    print(f"  Parameters: {num_params / 1e6:.1f}M")
    
    del model
    torch.cuda.empty_cache()
    
except Exception as e:
    print(f"  ✗ Error: {e}")
    print("\nCheck your HF_TOKEN if accessing gated models.")

## 4. Test Weights & Biases

In [ ]:
try:
    import wandb
    
    print("Weights & Biases Configuration")
    print("=" * 50)
    
    api_key = os.getenv("WANDB_API_KEY")
    project = os.getenv("WANDB_PROJECT", "llm-quant-lab")
    entity = os.getenv("WANDB_ENTITY", None)
    
    if api_key:
        print(f"  API Key: ✓ Set")
        print(f"  Project: {project}")
        print(f"  Entity: {entity or 'default'}")
        
        # Test login (optional - uncomment to verify)
        # wandb.login(key=api_key)
        # print("  ✓ Login successful")
    else:
        print("  API Key: ○ Not set (experiments will run in offline mode)")
        print("  Get your API key at: https://wandb.ai/authorize")

except ImportError:
    print("wandb not installed. Install with: pip install wandb")

## 5. Test Database Connection

In [ ]:
from sqlalchemy import create_engine, text

print("Database Configuration")
print("=" * 50)

db_host = os.getenv("POSTGRES_HOST", "localhost")
db_port = os.getenv("POSTGRES_PORT", "5432")
db_user = os.getenv("POSTGRES_USER")
db_pass = os.getenv("POSTGRES_PASSWORD")
db_name = os.getenv("POSTGRES_DB")

if all([db_user, db_pass, db_name]):
    db_url = f"postgresql://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}"
    print(f"  Host: {db_host}:{db_port}")
    print(f"  Database: {db_name}")
    
    try:
        engine = create_engine(db_url)
        with engine.connect() as conn:
            result = conn.execute(text("SELECT 1"))
            print("  ✓ Connection successful")
            
            # Check tables
            result = conn.execute(text(
                "SELECT table_name FROM information_schema.tables "
                "WHERE table_schema = 'public'"
            ))
            tables = [row[0] for row in result]
            print(f"  Tables: {', '.join(tables) if tables else 'None (run migrations)'}")
    except Exception as e:
        print(f"  ✗ Connection failed: {e}")
        print("  Make sure PostgreSQL is running (docker-compose up db)")
else:
    print("  ✗ Database credentials not configured")

## 6. Quick Quantization Test

In [ ]:
print("Quick Quantization Test")
print("=" * 50)

# Simple RTN quantization test
import torch

def simple_quantize(weight, bit_width=4):
    """Simple round-to-nearest quantization."""
    qmax = 2 ** (bit_width - 1) - 1
    qmin = -(2 ** (bit_width - 1))
    
    max_val = torch.max(torch.abs(weight))
    scale = max_val / qmax
    
    quantized = torch.clamp(torch.round(weight / scale), qmin, qmax)
    dequantized = quantized * scale
    
    error = torch.mean((weight - dequantized) ** 2).item()
    return dequantized, error

# Test
test_weight = torch.randn(128, 128)
quantized_weight, error = simple_quantize(test_weight, bit_width=4)

print(f"  Original weight shape: {test_weight.shape}")
print(f"  Quantization error (MSE): {error:.6f}")
print(f"  ✓ Quantization test passed")

## 7. Available Experiments

After setup is complete, you can run the following experiments:

| Notebook | Description |
|----------|-------------|
| `01_gptq_paper_reproduction.ipynb` | Reproduce GPTQ paper results (4-bit weight quantization) |
| `02_smoothquant_paper_reproduction.ipynb` | Reproduce SmoothQuant paper results (W8A8 quantization) |

### Paper References

- **GPTQ**: [Frantar et al., 2022](https://arxiv.org/abs/2210.17323) - Hessian-based weight quantization
- **SmoothQuant**: [Xiao et al., 2022](https://arxiv.org/abs/2211.10438) - Activation smoothing for W8A8

### Recommended Models for Testing

Start with smaller models for faster iteration:

1. `facebook/opt-125m` - 125M parameters, ~250MB
2. `facebook/opt-350m` - 350M parameters, ~700MB  
3. `facebook/opt-1.3b` - 1.3B parameters, ~2.6GB

For paper reproduction, also test:
- `facebook/opt-6.7b` - 6.7B parameters
- `bigscience/bloom-560m` - 560M parameters

In [ ]:
print("\n" + "=" * 50)
print("Setup Complete!")
print("=" * 50)
print("\nNext steps:")
print("1. If running locally: Start containers with 'docker-compose up -d'")
print("2. Open 01_gptq_paper_reproduction.ipynb to run GPTQ experiments")
print("3. Open 02_smoothquant_paper_reproduction.ipynb for SmoothQuant")
print("\nHappy quantizing! 🚀")